# Tips for task 2a) in worksheet 02

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 1</h2>
    <details>
    <summary>Click here!</summary>
    
  When implementing your cost function ensure, that your calculated loss does directly depend on the output of your qnode. ArrayBoxes are used for automatic differentiation. And in case there is no direct dependency, information are lost and differentiation does fail. PennyLane will usually output a warning that the output does not depend on your input. This is usually a hint, that something might be wrong about your cost function.

  For your calculations within the cost function take care to not directly use numpy but the numpy API being delivered by PennyLane instead. You can call it like this:

  ```python
  pnp.mean(...)
  ```

</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 2</h2>
    <details>
    <summary>Click here!</summary>
    
  One possible implementation of your cost function might look like the following:

  ```python
  def cost(weights, features, labels):
    """
    Compute the binary cross-entropy loss.
    """
    # Ensure predictions are interpreted as probabilities, otherwise they are in the range [-1,1]
    predictions = pnp.array([(1 - vqc_classifier(feature, weights)()) / 2.0 for feature in features])
    # Compute binary cross-entropy loss
    loss = -pnp.mean(labels * pnp.log(predictions + 1e-10) + (1 - labels) * pnp.log(1 - predictions + 1e-10))
    return loss
  ```

</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 3</h2>
    <details>
    <summary>Click here!</summary>
    
  The weights will be trained during optimization. This requires to add the parameter `requires_grad=True`. This is provided by the numpy PennyLane interface. So take care to use this API instead of the common numpy. E.g. we use this initialization for our variational setup with three trainable gates per layer and qubit:

  ```python
  weights = pnp.random.rand(n_layers, n_qubits, 3, requires_grad=True)
  ```

</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 4</h2>
    <details>
    <summary>Click here!</summary>
    
  You can calculate the cost and the updated weights with a function [`step_and_cost()`](https://docs.pennylane.ai/en/stable/code/api/pennylane.GradientDescentOptimizer.html#pennylane.GradientDescentOptimizer.step_and_cost) provided by the PennyLane optimizers. The returned cost ist the cost prior to updating the weights.

  ```python
  weights, current_cost = optimizer.step_and_cost(lambda w: cost(w, X_scaled, y), weights)
  ```
  
  In this example, we are using a cost function named `cost` that accepts as parameters the weights, features, and labels.
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 5</h2>
    <details>
    <summary>Click here!</summary>
    
  To speed up training, you can use batches of e.g. $50$ training samples, so that the optimization does not run over all samples.

  ```python
  idx = np.random.choice(len(X_scaled), size=50, replace=False)
  Xb, yb = X_scaled[idx], y[idx]
  ```
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 6</h2>
    <details>
    <summary>Click here!</summary>
    
  Common pitfalls include the wrong scaling of input data. Please ensure, that your data uses the full range $[0, \pi]$ when it is encoded. So consider going back to exercise 1a) to ensure, that you are scaling your inputs by $\pi$.
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 7</h2>
    <details>
    <summary>Click here!</summary>
    
  Did you use data reuploading to encode your input data?
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 8</h2>
    <details>
    <summary>Click here!</summary>
    
  The learning rate in quantum machine learning is commonly a factor of $2$ to $10$ higher than learning rates used in machine learning. There are several reasons for this, e.g. the big sensitivity to small parameter changes due to the non-linearity of quantum effects and the high-dimensionality of quantum systems but also noise and barren plateaus are a good reason to not get stuck in local minima.

  It is reasonable to start with a learning rate of $0.1$ to $0.5$. In case your model starts oscillating, acts instable, or noise governs your training, you should consider an adaptive method such as Adam optimizer, step decay, or quantum dropout for regularization.

</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 9</h2>
    <details>
    <summary>Click here!</summary>
    
  Try to increase the number of layers and probably also the number of samples being used for training to get to lower loss values and improve your decision boundary, e.g. a number of $6$ layers shows already pretty good results.
  
</details>
</div>

<div class="alert alert-success">
  <h2><i class="fas fa-check" style="font-size:36px"></i> &nbsp; Exemplary Solution </h2>
    <details>
    <summary>Click here!</summary>

  ```python
  def cost(weights, features, labels):
    """
    Compute the binary cross-entropy loss.
    """
    # Ensure predictions are interpreted as probabilities, otherwise they are in the range [-1,1]
    predictions = pnp.array([(1 - vqc_classifier(feature, weights)()) / 2.0 for feature in features])
    # Compute binary cross-entropy loss
    loss = -pnp.mean(labels * pnp.log(predictions + 1e-10) + (1 - labels) * pnp.log(1 - predictions + 1e-10))
    return loss

  # Step 2: Initialize weights
  n_qubits = 2
  n_layers = 6
  weights = pnp.random.rand(n_layers, n_qubits, 3, requires_grad=True)

  # Step 3: Training loop
  n_epochs = 100
  learning_rate = 0.1
  costs = [cost(weights, X_scaled, y)]
  print(f"Initial cost: {costs[0]:.4f}")

  optimizer = qml.GradientDescentOptimizer(learning_rate)

  for epoch in range(n_epochs):
    idx = np.random.choice(len(X_scaled), size=60, replace=False)
    Xb, yb = X_scaled[idx], y[idx]

    weights, current_cost = optimizer.step_and_cost(lambda w: cost(w, Xb, yb), weights)
    costs.append(current_cost)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{n_epochs}, Cost: {current_cost:.4f}")

  print("Training complete!")
  print(f"Final cost: {costs[-1]:.4f}")
  ```
</details>
</div>